# 04 — Supervised Fine-Tuning (SFT) Baseline
**Goal**: Fine-tune the 1.3B base model on APPS problem-solution pairs using LoRA adapters. Produces the `./checkpoints/sft/final` adapter checkpoint to warm-start RL training (PPO & DPO).

---

## Step 1: Environment & Universal Path Resolution

In [ ]:
import sys, os, shutil

# Universal Path Resolution & Auto-Copy for Kaggle / Local / Colab
def prepare_kaggle_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    # Search /kaggle/input for src folder
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(root, dest)
                print(f"Copied 'src' from {root} to {dest}")
                return '/kaggle/working'
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                dest = '/kaggle/working/src'
                if os.path.exists(dest):
                    shutil.rmtree(dest)
                shutil.copytree(src_dir, dest)
                print(f"Copied 'src' from {src_dir} to {dest}")
                return '/kaggle/working'
    
    # Fallback to parent directory
    parent = os.path.abspath('..')
    if os.path.exists(os.path.join(parent, 'src')):
        return parent
    return curr

repo_root = prepare_kaggle_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.training.sft import format_for_sft, run_sft_training

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Load APPS Dataset & Format for SFT

In [ ]:
print("Loading APPS training dataset via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:2000]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0)
print(f"Loaded {len(apps_clean)} APPS problem-solution pairs.")

sample_formatted = format_for_sft(apps_clean[0])
print("\n--- Sample Formatted SFT Prompt ---")
print(sample_formatted['text'][:300] + "...")

## Step 3: Load Base Model & Run SFT Training

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading base model {MODEL_NAME} for SFT in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(model_name=MODEL_NAME, load_in_4bit=False, lora_r=16)

print("\nStarting SFT Training loop...")
trainer = run_sft_training(
    model=model,
    tokenizer=tokenizer,
    dataset=apps_clean,
    output_dir="./checkpoints/sft",
    num_epochs=1,
    per_device_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
)

print("\nSFT Training completed successfully!")
print("Saved fine-tuned adapter to ./checkpoints/sft/final")

## Step 4: Checkpoint Verification & Inference Test
Verifies adapter files, reloads saved LoRA weights, confirms `lora_alpha=32` & target modules `['q_proj', 'v_proj']`, and executes 3 APPS inference tests.

In [ ]:
import os
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dir = None
search_roots = ['./checkpoints/sft/final', '/kaggle/working/checkpoints/sft/final', '/kaggle/working', '/kaggle/input']

for root in search_roots:
    if os.path.exists(root):
        if os.path.exists(os.path.join(root, 'adapter_config.json')):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if 'adapter_config.json' in files:
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir:
    raise FileNotFoundError("Could not locate adapter_config.json in /kaggle/working or /kaggle/input")

print(f"=== Found Checkpoint Directory: {checkpoint_dir} ===")
for fname in sorted(os.listdir(checkpoint_dir)):
    fpath = os.path.join(checkpoint_dir, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  - {fname}: {size_mb:.2f} MB")

print("\n=== 2. Verifying LoRA Configuration ===")
config = PeftConfig.from_pretrained(checkpoint_dir)
print(f"  - lora_alpha: {config.lora_alpha}")
print(f"  - r: {config.r}")
print(f"  - target_modules: {list(config.target_modules)}")
print(f"  - peft_type: {config.peft_type}")

print("\n=== 3. Reloading Saved SFT Model & Adapter ===")
base_model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/deepseek-coder-1.3b-instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
reloaded_model = PeftModel.from_pretrained(base_model, checkpoint_dir)
reloaded_model.eval()
print("Reloaded SFT adapter model successfully!")

print("\n=== 4. Running APPS Inference Check (3 Examples) ===")
sample_prompts = [
    apps_clean[0]['question'],
    apps_clean[1]['question'],
    apps_clean[2]['question']
]

for idx, p in enumerate(sample_prompts):
    prompt_text = f"### Problem:\n{p[:300]}\n\n### Solution:\n```python\n"
    inputs = tokenizer(prompt_text, return_tensors="pt").to(reloaded_model.device)
    with torch.no_grad():
        out = reloaded_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"\n--- Inference Sample {idx + 1} ---")
    print(gen_text[:250] + "...")

print("\nVerification completed successfully! SFT checkpoint is 100% functional.")